In [9]:
from langchain.tools import tool, BaseTool
from typing import TypedDict, Type
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from pydantic import Field, BaseModel

In [10]:
x = load_dotenv()

In [11]:
class Fun(TypedDict):
    num: float = Field(description="A random Number generated by user")
    chat_guess: float = Field(description='Guess of the LLM')
    is_greater: bool = Field(description='Flag for greater and less')
    

In [12]:
graph = StateGraph(Fun)

In [13]:
@tool
def compare_guess(guess: float, target: float) -> dict:
    """Compare a guess with the target number."""
    if guess < target:
        return {"is_greater": False, "message": "Too low"}
    if guess > target:
        return {"is_greater": True, "message": "Too high"}
    return {"is_greater": False, "message": "Correct"}


def llm_guess(state: Fun) -> dict:
    guess = state["num"] / 2
    evaluation = compare_guess.invoke({"guess": guess, "target": state["num"]})
    return {
        "chat_guess": guess,
        "is_greater": evaluation["is_greater"]
    }


In [14]:
graph.add_node('llm_guess', llm_guess)